# Recalibration and Drift

**Use case:** After embedding-space mapping, similarity scores are systematically compressed (margins shrink ~39%). The recalibration module corrects this drift so downstream absolute-score thresholds remain meaningful.

**When you'd reach for this:** Your application uses absolute score thresholds (e.g., "accept if similarity > 0.7") and you've migrated via isotrieve. Without recalibration, these thresholds silently become too conservative.

**What you need installed:** `isotrieve`, `numpy`, `scikit-learn`.

**Estimated runtime:** ~2 minutes.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/krish1925/isotrieve/blob/main/isotrieve-python/notebooks/06_recalibration_and_drift.ipynb)

In [ ]:
!pip install -q isotrieve numpy scikit-learn

In [ ]:
import numpy as np

from isotrieve import RidgeMapping
from isotrieve.recalibration import ScoreRecalibrator

print("imports OK")

## 1. The problem: score compression after mapping

When you map embeddings from one space to another, cosine similarities between paired vectors shrink. The "ceiling" (what you'd get with a perfect mapping) is always higher than what the linear map achieves.

This means:
- Scores that were 0.85 in the original space might become 0.72 after mapping
- A threshold of 0.75 that worked before now rejects good results
- The drift is **systematic**, not random — it can be corrected

In [ ]:
rng = np.random.default_rng(42)

N = 2000
D = 384  # same dims — recalibration is about score compression, not dim change
LATENT = 64

latent = rng.normal(size=(N, LATENT))
W_old = rng.normal(size=(LATENT, D)) / np.sqrt(LATENT)
W_new = rng.normal(size=(LATENT, D)) / np.sqrt(LATENT)  # different projection = different geometry

X_old = (latent @ W_old)
X_old = X_old / np.linalg.norm(X_old, axis=1, keepdims=True)
Y_new = (latent @ W_new)
Y_new = Y_new / np.linalg.norm(Y_new, axis=1, keepdims=True)

# Fit mapping
split = 1500
mapping = RidgeMapping(alpha="auto", seed=0)
mapping.fit(X_old[:split], Y_new[:split])

# Transform held-out vectors
X_hold = X_old[split:]
Y_hold = Y_new[split:]
X_mapped = mapping.transform(X_hold)

# Compare score distributions
ceiling_sims = np.sum(X_hold * Y_hold, axis=1)  # ideal: old vs new directly
mapped_sims = np.sum(X_mapped * Y_hold, axis=1)  # after mapping

print("Score distribution comparison:")
print(f"  Ceiling (ideal):  mean={ceiling_sims.mean():.4f}, p50={np.median(ceiling_sims):.4f}")
print(f"  After mapping:    mean={mapped_sims.mean():.4f}, p50={np.median(mapped_sims):.4f}")
print(f"  Drift (mean):     {ceiling_sims.mean() - mapped_sims.mean():.4f}")

## 2. Fit a ScoreRecalibrator

The recalibrator learns a monotone mapping from mapped scores → ceiling scores using isotonic regression. It needs paired (mapped_score, ceiling_score) examples.

In [ ]:
recal = ScoreRecalibrator()
recal.fit(mapped_sims, ceiling_sims)

report = recal.report
print("Recalibration report:")
print(f"  Pairs used: {report.n_pairs}")
print(f"  Mean mapped score:  {report.mean_mapped_score:.4f}")
print(f"  Mean ceiling score: {report.mean_ceiling_score:.4f}")
print(f"  Mean shift:         {report.mean_shift:+.4f}")
print(f"  Margin ratio:       {report.margin_ratio:.4f}")
print("\n  Threshold agreement (recalibrated vs ceiling):")
for tau, agree in report.threshold_agreement.items():
    print(f"    tau={tau}: {agree:.1%}")

## 3. Before vs After recalibration

In [ ]:
# Apply recalibration
recalibrated_sims = recal.transform(mapped_sims)

print("Score comparison:")
print(f"  {'':>20} {'mean':>8} {'median':>8} {'p5':>8} {'p95':>8}")
print(f"  {'Ceiling (ideal)':>20} {ceiling_sims.mean():>8.4f} {np.median(ceiling_sims):>8.4f} "
      f"{np.percentile(ceiling_sims, 5):>8.4f} {np.percentile(ceiling_sims, 95):>8.4f}")
print(f"  {'After mapping':>20} {mapped_sims.mean():>8.4f} {np.median(mapped_sims):>8.4f} "
      f"{np.percentile(mapped_sims, 5):>8.4f} {np.percentile(mapped_sims, 95):>8.4f}")
print(f"  {'After recalibration':>20} {recalibrated_sims.mean():>8.4f} {np.median(recalibrated_sims):>8.4f} "
      f"{np.percentile(recalibrated_sims, 5):>8.4f} {np.percentile(recalibrated_sims, 95):>8.4f}")

## 4. Impact on threshold-based decisions

In [ ]:
threshold = 0.75

# How many pairs pass the threshold under each scoring?
pass_ceiling = (ceiling_sims >= threshold).sum()
pass_mapped = (mapped_sims >= threshold).sum()
pass_recal = (recalibrated_sims >= threshold).sum()

print(f"Threshold: {threshold}")
print(f"  Ceiling (ideal):  {pass_ceiling}/{len(ceiling_sims)} pairs pass ({pass_ceiling/len(ceiling_sims):.1%})")
print(f"  After mapping:    {pass_mapped}/{len(mapped_sims)} pairs pass ({pass_mapped/len(mapped_sims):.1%})")
print(f"  After recal:      {pass_recal}/{len(recalibrated_sims)} pairs pass ({pass_recal/len(recalibrated_sims):.1%})")
print(f"\nWithout recalibration, you'd reject {pass_ceiling - pass_mapped} good results.")
print(f"Recalibration recovers {pass_recal - pass_mapped if pass_recal > pass_mapped else 0} of those.")

## 5. Monotonicity check

The recalibrator must be monotone: higher mapped scores → higher calibrated scores. This is guaranteed by isotonic regression, but let's verify.

In [ ]:
# Test monotonicity on a sorted grid
test_scores = np.linspace(0, 1, 100)
recalibrated = recal.transform(test_scores)

diffs = np.diff(recalibrated)
is_monotone = np.all(diffs >= -1e-10)  # allow tiny numerical noise
print(f"Monotone: {is_monotone}")
print(f"Max negative diff: {diffs.min():.6f}" if not is_monotone else "All diffs >= 0")

## 6. Save and reload

In [ ]:
recal.save("recalibrator.json")
recal2 = ScoreRecalibrator.load("recalibrator.json")

test = np.array([0.5, 0.6, 0.7, 0.8, 0.9])
original = recal.transform(test)
loaded = recal2.transform(test)

print(f"Round-trip OK: {np.allclose(original, loaded)}")

## 7. When to recalibrate vs re-fit from scratch

| Scenario | Recommendation |
|----------|---------------|
| Scores drifted after initial mapping, same data distribution | **Recalibrate** — cheap, monotone correction |
| New domain data, distribution shifted significantly | **Re-fit** the mapping from scratch |
| Gate says WARN but scores are important | **Recalibrate** first, re-check gate |
| Gate says FAIL | **Re-fit** or re-embed — recalibration won't save a bad mapping |
| Retraining calibration set is expensive | **Recalibrate** as a bridge, plan re-fit later |

## Try it yourself

Change the `threshold` from 0.75 to 0.6. Does the gap between mapped and recalibrated shrink? Why?

```python
threshold = 0.6  # try this
```